In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


# =====================================================
# CONFIG
# =====================================================
DATA_DIR = Path(".")
OUT_DIR = Path("anchor80_xgb20_outputs")
OUT_DIR.mkdir(exist_ok=True)

SALES_FILE = DATA_DIR / "sales.csv"
SAMPLE_FILE = DATA_DIR / "sample_submission.csv"
PROMO_FILE = DATA_DIR / "promotions.csv"

ANCHOR_WEIGHT = 0.8
ML_WEIGHT = 0.2
RANDOM_STATE = 42


# =====================================================
# LOAD DATA
# =====================================================
sales = pd.read_csv(SALES_FILE, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sample = pd.read_csv(SAMPLE_FILE, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

if PROMO_FILE.exists():
    promos = pd.read_csv(PROMO_FILE, parse_dates=["start_date", "end_date"])
else:
    promos = pd.DataFrame()


def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAPE": np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100,
        "R2": r2_score(y_true, y_pred),
    }


# =====================================================
# 1. SEASONAL-YOY ANCHOR
# =====================================================
def compute_yoy_growth(train_df):
    tmp = train_df.copy()
    tmp["year"] = tmp["Date"].dt.year

    yearly = (
        tmp.groupby("year", as_index=False)
        .agg(
            Revenue=("Revenue", "sum"),
            COGS=("COGS", "sum")
        )
        .sort_values("year")
    )

    yearly_full = yearly[(yearly["year"] >= 2013) & (yearly["year"] <= 2022)].copy()

    if len(yearly_full) < 3:
        yearly_full = yearly.copy()

    yearly_full["Revenue_yoy"] = yearly_full["Revenue"].pct_change()
    yearly_full["COGS_yoy"] = yearly_full["COGS"].pct_change()

    revenue_growth = yearly_full["Revenue_yoy"].replace([np.inf, -np.inf], np.nan).dropna().mean()
    cogs_growth = yearly_full["COGS_yoy"].replace([np.inf, -np.inf], np.nan).dropna().mean()

    if pd.isna(revenue_growth):
        revenue_growth = 0.0

    if pd.isna(cogs_growth):
        cogs_growth = 0.0

    revenue_growth = float(np.clip(revenue_growth, -0.20, 0.20))
    cogs_growth = float(np.clip(cogs_growth, -0.20, 0.20))

    return yearly_full, revenue_growth, cogs_growth


def build_seasonal_profile(train_df):
    tmp = train_df.copy()
    tmp["dayofyear"] = tmp["Date"].dt.dayofyear

    profile = (
        tmp.groupby("dayofyear", as_index=False)
        .agg(
            Revenue_profile=("Revenue", "mean"),
            COGS_profile=("COGS", "mean"),
            Revenue_profile_median=("Revenue", "median"),
            COGS_profile_median=("COGS", "median"),
        )
    )

    return profile


def add_anchor_forecast(date_df, train_df):
    df = date_df[["Date"]].copy()
    df["year"] = df["Date"].dt.year
    df["dayofyear"] = df["Date"].dt.dayofyear

    yearly_growth, revenue_growth, cogs_growth = compute_yoy_growth(train_df)
    seasonal_profile = build_seasonal_profile(train_df)

    last_train_year = int(train_df["Date"].dt.year.max())

    df = df.merge(seasonal_profile, on="dayofyear", how="left")

    df["Revenue_profile"] = df["Revenue_profile"].fillna(train_df["Revenue"].mean())
    df["COGS_profile"] = df["COGS_profile"].fillna(train_df["COGS"].mean())
    df["Revenue_profile_median"] = df["Revenue_profile_median"].fillna(train_df["Revenue"].median())
    df["COGS_profile_median"] = df["COGS_profile_median"].fillna(train_df["COGS"].median())

    df["years_after_train"] = df["year"] - last_train_year

    df["Revenue_yoy_growth_avg"] = revenue_growth
    df["COGS_yoy_growth_avg"] = cogs_growth

    df["Revenue_year_scale"] = (1 + revenue_growth) ** df["years_after_train"]
    df["COGS_year_scale"] = (1 + cogs_growth) ** df["years_after_train"]

    df["Revenue_anchor_yoy"] = df["Revenue_profile"] * df["Revenue_year_scale"]
    df["COGS_anchor_yoy"] = df["COGS_profile"] * df["COGS_year_scale"]

    df["Revenue_anchor_yoy_median"] = df["Revenue_profile_median"] * df["Revenue_year_scale"]
    df["COGS_anchor_yoy_median"] = df["COGS_profile_median"] * df["COGS_year_scale"]

    return df, yearly_growth, seasonal_profile


# =====================================================
# 2. FEATURE ENGINEERING
# =====================================================
def add_calendar_features(df):
    df = df.copy()
    d = df["Date"]

    df["year"] = d.dt.year
    df["month"] = d.dt.month
    df["day"] = d.dt.day
    df["dayofweek"] = d.dt.dayofweek
    df["dayofyear"] = d.dt.dayofyear
    df["weekofyear"] = d.dt.isocalendar().week.astype(int)
    df["quarter"] = d.dt.quarter

    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["is_month_start"] = d.dt.is_month_start.astype(int)
    df["is_month_end"] = d.dt.is_month_end.astype(int)
    df["is_payday_endmonth"] = (df["day"] >= 28).astype(int)

    df["is_double_day"] = ((df["month"] == df["day"]) & (df["day"] <= 12)).astype(int)
    df["is_11_11"] = ((df["month"] == 11) & (df["day"] == 11)).astype(int)
    df["is_12_12"] = ((df["month"] == 12) & (df["day"] == 12)).astype(int)
    df["is_year_end_sale"] = ((df["month"] == 12) & (df["day"] >= 15)).astype(int)
    df["is_black_friday_period"] = ((df["month"] == 11) & (df["day"].between(20, 30))).astype(int)

    df["time_idx"] = np.arange(len(df))
    df["time_idx_sq"] = df["time_idx"] ** 2

    for k in range(1, 7):
        df[f"sin_year_{k}"] = np.sin(2 * np.pi * k * df["dayofyear"] / 365.25)
        df[f"cos_year_{k}"] = np.cos(2 * np.pi * k * df["dayofyear"] / 365.25)

    for k in range(1, 3):
        df[f"sin_week_{k}"] = np.sin(2 * np.pi * k * df["dayofweek"] / 7)
        df[f"cos_week_{k}"] = np.cos(2 * np.pi * k * df["dayofweek"] / 7)

    return df


def add_promo_features(df):
    df = df.copy()

    if promos.empty:
        df["active_promo_count"] = 0
        df["avg_discount_value"] = 0
        df["max_discount_value"] = 0
        df["percentage_promo_count"] = 0
        df["fixed_promo_count"] = 0
        df["stackable_promo_count"] = 0
        return df

    active_count = []
    avg_discount = []
    max_discount = []
    percentage_count = []
    fixed_count = []
    stackable_count = []

    for dt in df["Date"]:
        active = promos[(promos["start_date"] <= dt) & (promos["end_date"] >= dt)]

        active_count.append(len(active))
        avg_discount.append(active["discount_value"].mean() if len(active) else 0)
        max_discount.append(active["discount_value"].max() if len(active) else 0)
        percentage_count.append((active["promo_type"] == "percentage").sum() if len(active) else 0)
        fixed_count.append((active["promo_type"] == "fixed").sum() if len(active) else 0)
        stackable_count.append(active["stackable_flag"].sum() if len(active) else 0)

    df["active_promo_count"] = active_count
    df["avg_discount_value"] = avg_discount
    df["max_discount_value"] = max_discount
    df["percentage_promo_count"] = percentage_count
    df["fixed_promo_count"] = fixed_count
    df["stackable_promo_count"] = stackable_count

    return df


def build_features(train_df, future_dates):
    future = pd.DataFrame({
        "Date": pd.to_datetime(future_dates),
        "Revenue": np.nan,
        "COGS": np.nan
    })

    df = pd.concat(
        [train_df[["Date", "Revenue", "COGS"]], future],
        ignore_index=True
    ).sort_values("Date").reset_index(drop=True)

    df = add_calendar_features(df)
    df = add_promo_features(df)

    anchor_df, yearly_growth, seasonal_profile = add_anchor_forecast(df[["Date"]], train_df)

    anchor_cols = [
        "Date",
        "Revenue_profile",
        "COGS_profile",
        "Revenue_profile_median",
        "COGS_profile_median",
        "Revenue_yoy_growth_avg",
        "COGS_yoy_growth_avg",
        "Revenue_year_scale",
        "COGS_year_scale",
        "Revenue_anchor_yoy",
        "COGS_anchor_yoy",
        "Revenue_anchor_yoy_median",
        "COGS_anchor_yoy_median",
    ]

    df = df.merge(anchor_df[anchor_cols], on="Date", how="left")

    for col in ["Revenue", "COGS"]:
        for lag in [365, 366, 728, 729, 730, 731, 732, 1092, 1095]:
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)

        base = df[col].shift(730)

        for w in [7, 14, 28, 60, 90, 180, 365]:
            df[f"{col}_lag730_roll_mean_{w}"] = base.rolling(
                w, min_periods=max(2, w // 3)
            ).mean()

            df[f"{col}_lag730_roll_std_{w}"] = base.rolling(
                w, min_periods=max(2, w // 3)
            ).std()

            df[f"{col}_lag730_roll_median_{w}"] = base.rolling(
                w, min_periods=max(2, w // 3)
            ).median()

        df[f"{col}_lag730_ewm_30"] = base.ewm(span=30, adjust=False).mean()
        df[f"{col}_lag730_ewm_90"] = base.ewm(span=90, adjust=False).mean()

        df[f"{col}_yoy_growth_365_730"] = (
            df[f"{col}_lag_365"] / (df[f"{col}_lag_730"] + 1e-6) - 1
        )

        df[f"{col}_yoy_growth_730_1095"] = (
            df[f"{col}_lag_730"] / (df[f"{col}_lag_1095"] + 1e-6) - 1
        )

        df[f"{col}_baseline_lag730_yoy"] = (
            df[f"{col}_lag_730"] *
            (1 + df[f"{col}_yoy_growth_730_1095"].clip(-0.5, 0.5))
        )

    df["gross_margin_lag730"] = (
        (df["Revenue_lag_730"] - df["COGS_lag_730"]) /
        (df["Revenue_lag_730"] + 1e-6)
    )

    df["cogs_to_revenue_lag730"] = (
        df["COGS_lag_730"] / (df["Revenue_lag_730"] + 1e-6)
    )

    hist = df[df["Revenue"].notna()].copy()

    for col in ["Revenue", "COGS"]:
        for keys, name in [
            (["dayofyear"], "dayofyear"),
            (["month"], "month"),
            (["dayofweek"], "dayofweek"),
            (["weekofyear"], "weekofyear"),
            (["month", "dayofweek"], "month_dayofweek"),
        ]:
            mean_map = hist.groupby(keys)[col].mean().rename(
                f"{col}_hist_mean_{name}"
            ).reset_index()

            median_map = hist.groupby(keys)[col].median().rename(
                f"{col}_hist_median_{name}"
            ).reset_index()

            df = df.merge(mean_map, on=keys, how="left")
            df = df.merge(median_map, on=keys, how="left")

    return df, yearly_growth, seasonal_profile


def make_xgb_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=1200,
            learning_rate=0.02,
            max_depth=3,
            min_child_weight=10,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.5,
            reg_lambda=10.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
        )
    )


# =====================================================
# 3. VALIDATION: LAST 548 DAYS
# =====================================================
horizon = len(sample)
val_start = sales["Date"].iloc[-horizon]

train_part = sales[sales["Date"] < val_start].copy()
val_part = sales[sales["Date"] >= val_start].copy()

features_val, yearly_growth, seasonal_profile = build_features(train_part, val_part["Date"])

train_feat = features_val[features_val["Date"] < val_start].copy()
val_feat = features_val[features_val["Date"] >= val_start].copy()

feature_cols = [
    c for c in train_feat.columns
    if c not in ["Date", "Revenue", "COGS"]
    and pd.api.types.is_numeric_dtype(train_feat[c])
]

feature_cols = [
    c for c in feature_cols
    if train_feat[c].notna().sum() >= 80
    and train_feat[c].nunique(dropna=True) > 1
]

print("Feature count:", len(feature_cols))

metrics_rows = []
val_pred = val_part[["Date", "Revenue", "COGS"]].copy()

for target in ["Revenue", "COGS"]:
    train_mask = (
        (train_feat["Date"] >= train_feat["Date"].min() + pd.Timedelta(days=1095)) &
        train_feat[target].notna()
    )

    X_train = train_feat.loc[train_mask, feature_cols]
    y_train = np.log1p(train_feat.loc[train_mask, target])

    X_val = val_feat[feature_cols]
    y_val = val_part[target].values

    model = make_xgb_model()
    model.fit(X_train, y_train)

    pred_xgb = np.expm1(model.predict(X_val))
    pred_xgb = np.clip(pred_xgb, 0, None)

    pred_anchor = val_feat[f"{target}_anchor_yoy"].values

    pred_final = ANCHOR_WEIGHT * pred_anchor + ML_WEIGHT * pred_xgb
    pred_final = np.clip(pred_final, 0, None)

    val_pred[f"{target}_anchor_yoy"] = pred_anchor
    val_pred[f"{target}_xgboost"] = pred_xgb
    val_pred[f"{target}_final"] = pred_final

    for model_name, pred in [
        ("seasonal_yoy_anchor", pred_anchor),
        ("xgboost_feature_model", pred_xgb),
        ("anchor80_xgb20", pred_final),
    ]:
        m = calc_metrics(y_val, pred)
        m["target"] = target
        m["model"] = model_name
        m["n_features"] = len(feature_cols)
        metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df = metrics_df[["target", "model", "n_features", "MAE", "RMSE", "MAPE", "R2"]]

metrics_df.to_csv(OUT_DIR / "validation_metrics.csv", index=False)
val_pred.to_csv(OUT_DIR / "validation_predictions.csv", index=False)

pd.DataFrame({
    "feature_no": range(1, len(feature_cols) + 1),
    "feature_name": feature_cols
}).to_csv(OUT_DIR / "feature_list.csv", index=False)

yearly_growth.to_csv(OUT_DIR / "yoy_growth_by_year.csv", index=False)
seasonal_profile.to_csv(OUT_DIR / "seasonal_profile.csv", index=False)

print(metrics_df)


# =====================================================
# 4. FINAL TRAIN + TEST PREDICTION
# =====================================================
features_test, yearly_growth_full, seasonal_profile_full = build_features(sales, sample["Date"])

train_full = features_test[features_test["Revenue"].notna()].copy()
test_feat = features_test[features_test["Date"].isin(sample["Date"])].copy()

submission = sample[["Date"]].copy()
test_feature_table = test_feat[["Date"]].copy()

for target in ["Revenue", "COGS"]:
    train_mask = (
        (train_full["Date"] >= train_full["Date"].min() + pd.Timedelta(days=1095)) &
        train_full[target].notna()
    )

    X_train = train_full.loc[train_mask, feature_cols]
    y_train = np.log1p(train_full.loc[train_mask, target])

    X_test = test_feat[feature_cols]

    model = make_xgb_model()
    model.fit(X_train, y_train)

    pred_xgb = np.expm1(model.predict(X_test))
    pred_xgb = np.clip(pred_xgb, 0, None)

    pred_anchor = test_feat[f"{target}_anchor_yoy"].values

    pred_final = ANCHOR_WEIGHT * pred_anchor + ML_WEIGHT * pred_xgb
    pred_final = np.clip(pred_final, 0, None)

    submission[target] = pred_final

    test_feature_table[f"{target}_anchor_yoy"] = pred_anchor
    test_feature_table[f"{target}_xgboost_pred"] = pred_xgb
    test_feature_table[f"{target}_final_pred"] = pred_final

submission.to_csv(OUT_DIR / "submission_anchor80_xgb20.csv", index=False)
test_feature_table.to_csv(OUT_DIR / "test_feature_table.csv", index=False)

monthly = submission.set_index("Date")[["Revenue", "COGS"]].resample("MS").sum().reset_index()
monthly.to_csv(OUT_DIR / "monthly_test_forecast.csv", index=False)


# =====================================================
# 5. VISUALIZATION
# =====================================================
plt.figure(figsize=(14, 5))
plt.plot(val_pred["Date"], val_pred["Revenue"], label="Actual Revenue")
plt.plot(val_pred["Date"], val_pred["Revenue_anchor_yoy"], label="Seasonal-YoY Anchor")
plt.plot(val_pred["Date"], val_pred["Revenue_xgboost"], label="XGBoost")
plt.plot(val_pred["Date"], val_pred["Revenue_final"], label="Final: 80% Anchor + 20% XGBoost")
plt.title("Validation Revenue: Anchor vs XGBoost vs Final")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "01_validation_revenue_anchor_xgb_final.png", dpi=180)
plt.close()

plt.figure(figsize=(14, 5))
plt.plot(val_pred["Date"], val_pred["COGS"], label="Actual COGS")
plt.plot(val_pred["Date"], val_pred["COGS_anchor_yoy"], label="Seasonal-YoY Anchor")
plt.plot(val_pred["Date"], val_pred["COGS_xgboost"], label="XGBoost")
plt.plot(val_pred["Date"], val_pred["COGS_final"], label="Final: 80% Anchor + 20% XGBoost")
plt.title("Validation COGS: Anchor vs XGBoost vs Final")
plt.xlabel("Date")
plt.ylabel("COGS")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_validation_cogs_anchor_xgb_final.png", dpi=180)
plt.close()

plt.figure(figsize=(14, 5))
plt.plot(sales["Date"], sales["Revenue"], alpha=0.65, label="Historical Revenue")
plt.plot(submission["Date"], submission["Revenue"], label="Test Revenue Forecast")
plt.axvspan(submission["Date"].min(), submission["Date"].max(), alpha=0.12, label="Test period")
plt.title("Test Forecast Revenue: 80% Seasonal-YoY + 20% XGBoost")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "03_test_forecast_revenue.png", dpi=180)
plt.close()

plt.figure(figsize=(14, 5))
plt.plot(sales["Date"], sales["COGS"], alpha=0.65, label="Historical COGS")
plt.plot(submission["Date"], submission["COGS"], label="Test COGS Forecast")
plt.axvspan(submission["Date"].min(), submission["Date"].max(), alpha=0.12, label="Test period")
plt.title("Test Forecast COGS: 80% Seasonal-YoY + 20% XGBoost")
plt.xlabel("Date")
plt.ylabel("COGS")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "04_test_forecast_cogs.png", dpi=180)
plt.close()

plt.figure(figsize=(14, 5))
plt.plot(monthly["Date"], monthly["Revenue"], marker="o", label="Monthly Revenue")
plt.plot(monthly["Date"], monthly["COGS"], marker="o", label="Monthly COGS")
plt.title("Monthly Test Forecast Totals")
plt.xlabel("Month")
plt.ylabel("Total")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "05_monthly_test_forecast.png", dpi=180)
plt.close()

plt.figure(figsize=(14, 5))
plt.plot(test_feature_table["Date"], test_feature_table["Revenue_anchor_yoy"], label="Revenue Anchor")
plt.plot(test_feature_table["Date"], test_feature_table["Revenue_xgboost_pred"], label="Revenue XGBoost")
plt.plot(test_feature_table["Date"], test_feature_table["Revenue_final_pred"], label="Revenue Final")
plt.title("Test Revenue: Anchor vs XGBoost vs Final")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "06_test_revenue_anchor_xgb_final.png", dpi=180)
plt.close()


# =====================================================
# 6. SUMMARY + ZIP
# =====================================================
summary = f"""
Anchor 80% + XGBoost 20% Forecast Summary

Data:
- sales rows: {len(sales)}
- sample rows: {len(sample)}
- validation horizon: {horizon}
- validation start: {val_start.date()}
- ML feature count: {len(feature_cols)}

Final formula:
Revenue_final = {ANCHOR_WEIGHT} * Revenue_seasonal_yoy_anchor + {ML_WEIGHT} * Revenue_XGBoost
COGS_final    = {ANCHOR_WEIGHT} * COGS_seasonal_yoy_anchor    + {ML_WEIGHT} * COGS_XGBoost

XGBoost:
- objective: reg:squarederror
- n_estimators: 1200
- learning_rate: 0.02
- max_depth: 3
- min_child_weight: 10
- subsample: 0.85
- colsample_bytree: 0.85
- reg_alpha: 0.5
- reg_lambda: 10.0

Main output:
- submission_anchor80_xgb20.csv
"""

(OUT_DIR / "run_summary.txt").write_text(summary, encoding="utf-8")

zip_path = Path("anchor80_xgb20_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.iterdir():
        z.write(p, arcname=f"anchor80_xgb20_outputs/{p.name}")

print("Done.")
print("Submission:", OUT_DIR / "submission_anchor80_xgb20.csv")
print("ZIP:", zip_path)
print(submission.head(10))

Feature count: 142
    target                  model  n_features           MAE          RMSE  \
0  Revenue    seasonal_yoy_anchor         142  1.221760e+06  1.533892e+06   
1  Revenue  xgboost_feature_model         142  5.705090e+05  7.772949e+05   
2  Revenue         anchor80_xgb20         142  1.009281e+06  1.289680e+06   
3     COGS    seasonal_yoy_anchor         142  9.512268e+05  1.213941e+06   
4     COGS  xgboost_feature_model         142  5.218495e+05  7.073692e+05   
5     COGS         anchor80_xgb20         142  7.841540e+05  1.014973e+06   

        MAPE        R2  
0  53.940704  0.024477  
1  21.870495  0.749493  
2  45.233778  0.310377  
3  45.414829  0.211749  
4  21.004961  0.732353  
5  37.804234  0.448965  
Done.
Submission: anchor80_xgb20_outputs/submission_anchor80_xgb20.csv
ZIP: anchor80_xgb20_outputs.zip
        Date       Revenue          COGS
0 2023-01-01  3.469410e+06  3.226782e+06
1 2023-01-02  1.743006e+06  1.510731e+06
2 2023-01-03  1.436773e+06  1.187763e+06